## Content-Based Recommender | Baseline Model

### Recommends based off the movies's plot, creditcs, genre and keywords via TFIDF Vectorization

###### Inspired by: Simulating Ibtesam
###### Link: https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system/notebook

In [ ]:
# Imports
import pandas as pd
import numpy as np
import random
from collections import Counter
from ast import literal_eval
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#%pip install sentence-transformers

In [3]:
# Read csv
df1=pd.read_csv('../tmdb/tmdb_5000_credits.csv')
df2=pd.read_csv('../tmdb/tmdb_5000_movies.csv')

In [4]:
# Join two datasets on id column
df1.columns = ['id','tittle','cast','crew']
df2= df2.merge(df1,on='id')

In [5]:
# Parse the stringified features into their corresponding python objects
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(literal_eval)

#### Functions that will help extract required info from each feature

In [6]:
# Get the director's name from the crew feature. If director is not listed, return NaN
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [7]:
# Test directors function
df2['director'] = df2['crew'].apply(get_director)
df2[['title', 'director']].head()

,title,director
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


In [8]:
# Returns the list top 3 elements or entire list; whichever is more.
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        
        # Check if more than 3 elements exist. If yes, return only first three. If no, return entire list.
        if len(names) > 3:
            names = names[:3]
        return names

    # Return empty list in case of missing/malformed data
    return []

In [9]:
# Define new director, cast, genres and keywords features that are in a suitable form.
df2['director'] = df2['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(get_list)

In [10]:
# Print the new features of the first 3 films
df2[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[culture clash, future, space war]","[Action, Adventure, Fantasy]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[ocean, drug abuse, exotic island]","[Adventure, Fantasy, Action]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[spy, based on novel, secret agent]","[Action, Adventure, Crime]"


#### Convert names and keywords instances into lowercase and strip spaces between them so vectorizer doesn't get confused by multiple people with the same first or last name.

In [11]:
# Function to convert all strings to lower case and strip names of spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        # Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [12]:
# Apply clean_data function to your features.
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df2[feature] = df2[feature].apply(clean_data)

## Building a unified text representation for each movie

In [ ]:
# Combine plot, cast, directory, genre, and keywords. Some are duplicate to balance the long length of the plot
# so they all have a comparable signal weight.
def unified_text(x):
    plot     = x['overview']              if isinstance(x['overview'],  str) else ''
    cast     = ' '.join(x['cast'])        if isinstance(x['cast'],     list) else ''
    genres   = ' '.join(x['genres'])      if isinstance(x['genres'],   list) else ''
    keywords = ' '.join(x['keywords'])    if isinstance(x['keywords'], list) else ''
    director = x['director']              if isinstance(x['director'],  str) else ''
    
    # Metadata terms repeated twice to balance against the longer plot text
    return f"{plot} {cast} {cast} {director} {director} {genres} {genres} {keywords} {keywords}".strip()

df2['overview'] = df2['overview'].fillna('')
df2['unified_text'] = df2.apply(unified_text, axis=1)

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
Name: unified_text, dtype: str

## Run Model | Needs to do TFIDF instead of Sentence Transformers

In [ ]:
# Build TF-IDF matrix and similarity
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df2['unified_text'])
cosine_sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(df2.index, index=df2['title']).drop_duplicates()

Loading weights: 100%|██████████| 103/103 [00:01<00:00, 52.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 76/76 [03:15<00:00,  2.58s/it]



Embeddings shape: (4803, 384)


## Recommendation function

In [ ]:
def get_recommendations(title):
    title = title.lower()

    indices_lower = indices.copy()
    indices_lower.index = indices_lower.index.str.lower()

    if title not in indices_lower:
        return f"Movie '{title}' not found in database."

    idx = indices_lower[title]
    sim_scores = list(enumerate(cosine_sim_tfidf[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]
    return df2['title'].iloc[movie_indices]

## Prepare for Evaluation

#### Test users and movies they enjoy

In [17]:
# dictionary of 5 users with difference preferences
# Each movie is considered a movie the user enjoyed

user_history = {
    # User 1: Classic gangster & crime dramas
    "user_1": [
        "The Godfather", "GoodFellas", "Scarface", "Pulp Fiction", "The Departed",
        "The Godfather: Part II", "Casino", "Donnie Brasco", "Once Upon a Time in America",
        "The Untouchables", "Road to Perdition", "Public Enemies", "Gangs of New York",
        "A History of Violence", "Eastern Promises", "The Town", "The Conformist",
        "Find Me Guilty", "Black Mass", "The Hills Have Eyes"
    ],

    # User 2: Blockbuster action & superhero movies
    "user_2": [
        "Avatar", "Titanic", "Avengers: Age of Ultron", "Guardians of the Galaxy", "Iron Man",
        "Thor", "Captain America: The First Avenger", "The Avengers", "Ant-Man",
        "The Incredible Hulk", "Captain America: Civil War", "Iron Man 2", "Iron Man 3",
        "Thor: The Dark World", "Batman Begins", "The Dark Knight", "The Dark Knight Rises",
        "Batman & Robin", "Batman Returns", "Batman v Superman: Dawn of Justice"
    ],

    # User 3: Musical & biographical movies
    "user_3": [
        "Chicago", "Moulin Rouge!", "8MM", "Amnesiac", "Grease",
        "Les Misérables", "Inception", "The Pursuit of Happyness", "The Hit List",
        "Singin' in the Rain", "The Sound of Music", "West Side Story", "Mary Poppins",
        "The Wizard of Oz", "Frozen", "Aladdin", "Cinderella", "The Nutcracker",
        "Alice in Wonderland", "The Broadway Melody"
    ],

    # User 4: Fantasy & young adult series
    "user_4": [
        "The Lord of the Rings: The Fellowship of the Ring", "The Hobbit: An Unexpected Journey",
        "The Lord of the Rings: The Two Towers", "The Lord of the Rings: The Return of the King",
        "Harry Potter and the Philosopher's Stone", "Harry Potter and the Chamber of Secrets",
        "Harry Potter and the Prisoner of Azkaban", "The Hunger Games: Catching Fire",
        "The Hunger Games: Mockingjay - Part 2", "The Twilight Saga: New Moon",
        "The Twilight Saga: Eclipse", "The Twilight Saga: Breaking Dawn - Part 2",
        "Percy Jackson: Sea of Monsters", "Percy Jackson & the Olympians: The Lightning Thief",
        "Harry Potter and the Order of the Phoenix", "The Chronicles of Narnia: The Lion, the Witch and the Wardrobe",
        "The Hobbit: The Desolation of Smaug", "The Hobbit: The Battle of the Five Armies",
        "The Adventures of Huck Finn", "Hellboy II: The Golden Army"
    ],

    # User 5: Horror & thriller movies
    "user_5": [
        "The Shining", "1408", "8 Days", "The Conjuring", "Insidious",
        "Sinister", "Annabelle", "Paranormal Activity 2", "Halloween: Resurrection", "Psycho",
        "Jaws", "Saw: The Final Chapter", "Scream 3", "Pet Sematary", "White Noise 2: The Light",
        "It Follows", "The Possession", "The Exorcist", "Evil Dead", "Restoration"
    ]
}

In [18]:
# Test get_recommendations for every liked movie of each user

counter = 0

for user, movies in user_history.items():
    #print(f"\nRecommendations for {user}:")
    for movie in movies:
        #print(f"\nMovie: {movie}")
        recs = get_recommendations(movie)
        if "not found" in recs:
            counter +=1
            print(recs)
            
print(f"\nTotal movies not found: {counter}")


Total movies not found: 0


## Split data into known and unknown

In [19]:
# Split movies into training and testing to evaluate model (5 training, 15 testing)
train_test_split = {}
split_ratio = 5

# Same results
random.seed(7)

# list and dict to store each unknown movie for each user
#users = {}

for user, movies in user_history.items():
    
    # List of unknown movies
    unknown_movies = []
    
    # add movies to either known or unknown list
    known = random.sample(movies, split_ratio)
    unknown = [m for m in movies if m not in known]
    
    # Add known and unknow movies to dictionary for corresponding user
    train_test_split[user] = {"known": known, "unknown": unknown}
    
    # append unknown movies to list
    #unknown_movies.append(unknown)
    #users[user] = unknown_movies

# Print 
for user, split in train_test_split.items():
    print(f"{user}:")
    print("Known:", split["known"])
    print("Unknown:", split["unknown"])
    print()

user_1:
Known: ['Road to Perdition', 'The Departed', 'Gangs of New York', 'GoodFellas', 'Scarface']
Unknown: ['The Godfather', 'Pulp Fiction', 'The Godfather: Part II', 'Casino', 'Donnie Brasco', 'Once Upon a Time in America', 'The Untouchables', 'Public Enemies', 'A History of Violence', 'Eastern Promises', 'The Town', 'The Conformist', 'Find Me Guilty', 'Black Mass', 'The Hills Have Eyes']

user_2:
Known: ['Batman & Robin', 'Guardians of the Galaxy', 'Iron Man 2', 'Titanic', 'Captain America: The First Avenger']
Unknown: ['Avatar', 'Avengers: Age of Ultron', 'Iron Man', 'Thor', 'The Avengers', 'Ant-Man', 'The Incredible Hulk', 'Captain America: Civil War', 'Iron Man 3', 'Thor: The Dark World', 'Batman Begins', 'The Dark Knight', 'The Dark Knight Rises', 'Batman Returns', 'Batman v Superman: Dawn of Justice']

user_3:
Known: ['Moulin Rouge!', '8MM', 'The Wizard of Oz', 'The Nutcracker', 'Alice in Wonderland']
Unknown: ['Chicago', 'Amnesiac', 'Grease', 'Les Misérables', 'Inception', 'T

## Evaluation

In [23]:
def evaluate(train_test_split, k=10):
    
    # Dict to store users and their precision score
    results = {}

    for user, data in train_test_split.items():
        
        # list of movies in unknown list that were discovered when recommending from known movies
        movies_found = []
        
        known_movies = data["known"]
        unknown_movies = data["unknown"]

        # List of all movies that were recommended for a users 5 known movies
        all_recommendations = []

        # Get recommendations for each known movie
        for movie in known_movies:
            recs = get_recommendations(movie)

            # If your function returns a string when not found, skip
            if isinstance(recs, str):
                continue

            # Convert to list and extend
            all_recommendations.extend(recs.tolist())

        movie_counts = Counter(all_recommendations)
        top_k = [movie for movie, _ in movie_counts.most_common(k)]

        # Count hits
        hits = sum(1 for movie in top_k if movie in unknown_movies)

        # Calsulate precision and add to dictionary
        precision = hits / k
        recall = hits / len(unknown_movies)
        results[user] = {"precision": precision, "recall": recall}
        
        # Add movies to list if they were in unknown_movies
        for movie in top_k:
            if movie in unknown_movies:
                movies_found.append(movie)

        # Print results
        print(f"{user}: Precision@{k} = {precision:.2f} ({hits}/{k}) | Recall@{k} = {recall:.2f} ({hits}/{len(unknown_movies)})")
        print()
        print(f"Top-{k} recommendations: {top_k}")
        print()
        print(f"Movies found in unknown set: {movies_found}")
        print()
        
    # Summary averages
    avg_precision = sum(v["precision"] for v in results.values()) / len(results)
    avg_recall = sum(v["recall"] for v in results.values()) / len(results)
    print(f"Average Precision@{k}: {avg_precision:.2f}")
    print(f"Average Recall@{k}:    {avg_recall:.2f}")

    # Return results
    return results

In [24]:
# Evaluate
evaluate(train_test_split, 10)

# Precision@K: How many recommended movies were actually liked (in unknown set).
# Recall@k: How many of the unknown liked movies did the model manage to recommend.

user_1: Precision@10 = 0.10 (1/10) | Recall@10 = 0.07 (1/15)

Top-10 recommendations: ['Donnie Brasco', 'Frequency', 'Snitch', 'Funny Games', 'Chicago Overcoat', 'One Hour Photo', 'Running Scared', 'Tinker Tailor Soldier Spy', 'Echo Dr.', 'Kiss Kiss Bang Bang']

Movies found in unknown set: ['Donnie Brasco']

user_2: Precision@10 = 0.60 (6/10) | Recall@10 = 0.40 (6/15)

Top-10 recommendations: ['Iron Man', 'The Dark Knight', 'Batman', 'Batman Forever', 'Batman v Superman: Dawn of Justice', 'Batman Returns', 'Batman Begins', 'The Dark Knight Rises', 'Superman IV: The Quest for Peace', 'Batman: The Dark Knight Returns, Part 2']

Movies found in unknown set: ['Iron Man', 'The Dark Knight', 'Batman v Superman: Dawn of Justice', 'Batman Returns', 'Batman Begins', 'The Dark Knight Rises']

user_3: Precision@10 = 0.10 (1/10) | Recall@10 = 0.07 (1/15)

Top-10 recommendations: ['Aladdin', 'As It Is in Heaven', 'The Cotton Club', 'The Cooler', 'Glitter', 'The Big Bounce', 'Idlewild', 'Take the L

{'user_1': {'precision': 0.1, 'recall': 0.06666666666666667},
 'user_2': {'precision': 0.6, 'recall': 0.4},
 'user_3': {'precision': 0.1, 'recall': 0.06666666666666667},
 'user_4': {'precision': 0.5, 'recall': 0.3333333333333333},
 'user_5': {'precision': 0.1, 'recall': 0.06666666666666667}}